# Roles module: Wearer / Trusted contact (SW341 · G3)
Run these cells **after** the Authentication cells (same notebook), because they reuse `app`, `db`, `User`, `session` and `log_security_event`.

| Area | Requirements |
|---|---|
| Contact ID | FR17, FR18, SR20, PR11 |
| Invitations | FR19, FR20, FR21, PR3, PR5, PR6 |
| Sharing control | FR22, FR23, FR24, PR4, PR7, PR10 |
| Authorization (401/403) | SR10, SR11, SR12, T7, T13 |
| Audit events | SR21, SR22 (no health values in logs) |

All sharing endpoints use the **link id** (not the user id), so no account ID or contact ID is exposed to other users (PR11).

In [ ]:
import re
import secrets
from datetime import datetime, timedelta, timezone
from functools import wraps

from flask import g
from markupsafe import escape
from sqlalchemy import UniqueConstraint, or_
from sqlalchemy.exc import IntegrityError

# ---------------- Constants ----------------
# 32-character alphabet without look-alike characters (0/O, 1/I): 32^8 possible IDs
CONTACT_ID_ALPHABET = "ABCDEFGHJKLMNPQRSTUVWXYZ23456789"
CONTACT_ID_LENGTH = 8                       # SR20
INVITATION_LIFETIME = timedelta(days=7)     # PR5
CATEGORIES = ("heart_rate_summary", "step_summary", "sleep_summary", "alerts")  # FR22


def utcnow():
    return datetime.now(timezone.utc)


def safe(text):
    """SR25: HTML-encode user-supplied text before it is returned for display."""
    return str(escape(text))


# ---------------- Models ----------------
class ContactProfile(db.Model):
    """FR17: one private contact ID per account."""
    __tablename__ = "contact_profiles"

    id = db.Column(db.Integer, primary_key=True)
    user_id = db.Column(db.Integer, db.ForeignKey("users.id"), unique=True, nullable=False)
    contact_id = db.Column(db.String(CONTACT_ID_LENGTH), unique=True, nullable=False)


class TrustedLink(db.Model):
    """Invitation + trusted-contact link + the categories the wearer selected."""
    __tablename__ = "trusted_links"
    __table_args__ = (UniqueConstraint("wearer_id", "contact_user_id"),)

    id = db.Column(db.Integer, primary_key=True)
    wearer_id = db.Column(db.Integer, db.ForeignKey("users.id"), nullable=False)
    contact_user_id = db.Column(db.Integer, db.ForeignKey("users.id"), nullable=False)
    status = db.Column(db.String(10), nullable=False, default="pending")   # pending | accepted
    created_at = db.Column(db.DateTime(timezone=True), default=utcnow, nullable=False)
    expires_at = db.Column(db.DateTime(timezone=True), nullable=True)      # only while pending
    accepted_at = db.Column(db.DateTime(timezone=True), nullable=True)

    # PR4: every category is False (unselected) by default
    share_heart_rate_summary = db.Column(db.Boolean, default=False, nullable=False)
    share_step_summary = db.Column(db.Boolean, default=False, nullable=False)
    share_sleep_summary = db.Column(db.Boolean, default=False, nullable=False)
    share_alerts = db.Column(db.Boolean, default=False, nullable=False)


with app.app_context():
    db.create_all()   # creates only the two new tables; existing tables are untouched
print("ContactProfile and TrustedLink tables are ready.")

In [ ]:
# ---------------- Contact ID (FR17, SR20) ----------------
def generate_contact_id():
    # secrets = cryptographically secure generator (SR20)
    return "".join(secrets.choice(CONTACT_ID_ALPHABET) for _ in range(CONTACT_ID_LENGTH))


def assign_contact_id(user_id):
    """Create the contact ID for an account (idempotent). Returns the ID."""
    existing = ContactProfile.query.filter_by(user_id=user_id).first()
    if existing:
        return existing.contact_id

    for _ in range(10):
        candidate = generate_contact_id()
        if ContactProfile.query.filter_by(contact_id=candidate).first() is None:
            db.session.add(ContactProfile(user_id=user_id, contact_id=candidate))
            try:
                db.session.commit()
                return candidate
            except IntegrityError:      # extremely unlikely collision: try again
                db.session.rollback()
    raise RuntimeError("Could not generate a unique contact ID")


# Assign the contact ID automatically when an account is created (FR17),
# without editing the Authentication cell.
if not getattr(app.view_functions["register"], "_has_contact_id_hook", False):
    _original_register = app.view_functions["register"]

    def register_with_contact_id():
        response = _original_register()
        body, status = response if isinstance(response, tuple) else (response, response.status_code)
        if status == 201:
            assign_contact_id(body.get_json()["user_id"])
        return response

    register_with_contact_id._has_contact_id_hook = True
    app.view_functions["register"] = register_with_contact_id


# ---------------- Authorization core (SR10, SR11, SR12) ----------------
def purge_expired_invitations():
    """PR5: delete invitations not accepted within 7 days."""
    TrustedLink.query.filter(
        TrustedLink.status == "pending",
        TrustedLink.expires_at < utcnow()
    ).delete(synchronize_session=False)
    db.session.commit()


def login_required(view):
    """SR11: HTTP 401 when there is no valid session."""
    @wraps(view)
    def wrapped(*args, **kwargs):
        user_id = session.get("user_id")
        user = db.session.get(User, user_id) if user_id is not None else None

        if user is None:
            if user_id is not None:
                session.clear()
            log_security_event("AUTH_REQUIRED")
            return jsonify({"error": "Authentication required"}), 401

        g.user = user
        purge_expired_invitations()
        return view(*args, **kwargs)
    return wrapped


def authorize_access(requester_id, owner_id, resource):
    """
    SR10: central check used on EVERY request for another user's data.
      - the owner can access their own data
      - "display_name": any user linked by a pending OR accepted invitation
      - a data category: only an ACCEPTED trusted contact AND only if the wearer selected it
    The database is queried on every call, so removal/deselection applies from the next request (PR7).
    """
    if requester_id == owner_id:
        return True

    if resource == "display_name":
        link = TrustedLink.query.filter(
            or_(
                (TrustedLink.wearer_id == owner_id) & (TrustedLink.contact_user_id == requester_id),
                (TrustedLink.wearer_id == requester_id) & (TrustedLink.contact_user_id == owner_id),
            )
        ).first()
        return link is not None

    if resource in CATEGORIES:
        link = TrustedLink.query.filter_by(
            wearer_id=owner_id, contact_user_id=requester_id, status="accepted"
        ).first()
        return bool(link and getattr(link, "share_" + resource))

    return False


def access_denied():
    """SR12 + SR21: HTTP 403 and an audit event (no health values in the log, SR22)."""
    log_security_event("AUTHORIZATION_FAILED", g.user.id)
    return jsonify({"error": "Access denied"}), 403


def categories_of(link):
    return {c: getattr(link, "share_" + c) for c in CATEGORIES}


def get_shared_summary(wearer_id, category):
    # TODO (wellness-processing teammate): return the wearer's DAILY summary for this category
    # (and, for "alerts", the alert list including the reading value). Authorization is already done by the caller.
    return {}

In [ ]:
# ---------------- Wearer: contact ID and invitations ----------------
@app.get("/api/roles/contact-id")
@login_required
def get_my_contact_id():
    # FR18: only the owner can see the contact ID
    return jsonify({"contact_id": assign_contact_id(g.user.id)}), 200


@app.post("/api/invitations")
@login_required
def send_invitation():
    # FR19: the wearer invites a registered user by that user's contact ID
    data = request.get_json(silent=True)
    if not isinstance(data, dict) or not isinstance(data.get("contact_id"), str):
        return jsonify({"error": "Invalid contact ID"}), 400

    cid = data["contact_id"].strip().upper()
    if len(cid) != CONTACT_ID_LENGTH or any(ch not in CONTACT_ID_ALPHABET for ch in cid):
        return jsonify({"error": "Invalid contact ID"}), 400

    profile = ContactProfile.query.filter_by(contact_id=cid).first()
    if profile is None:
        log_security_event("INVITATION_UNKNOWN_CONTACT_ID", g.user.id)
        return jsonify({"error": "Contact ID not found"}), 404

    if profile.user_id == g.user.id:
        return jsonify({"error": "You cannot invite yourself"}), 400

    if TrustedLink.query.filter_by(wearer_id=g.user.id, contact_user_id=profile.user_id).first():
        return jsonify({"error": "This user is already invited or linked"}), 409

    link = TrustedLink(
        wearer_id=g.user.id,
        contact_user_id=profile.user_id,
        status="pending",
        expires_at=utcnow() + INVITATION_LIFETIME,   # PR5
    )
    db.session.add(link)
    db.session.commit()
    log_security_event("INVITATION_SENT", g.user.id)

    invited = db.session.get(User, profile.user_id)
    return jsonify({
        "link_id": link.id,
        "contact_name": safe(invited.display_name),
        "status": link.status,
        "expires_at": link.expires_at.isoformat(),
    }), 201


# ---------------- Invited user: pending invitations ----------------
@app.get("/api/invitations/pending")
@login_required
def list_pending_invitations():
    # FR20: show pending invitations with the inviting wearer's display name only
    links = TrustedLink.query.filter_by(contact_user_id=g.user.id, status="pending").all()
    result = []
    for link in links:
        wearer = db.session.get(User, link.wearer_id)
        result.append({
            "invitation_id": link.id,
            "wearer_name": safe(wearer.display_name),
            "expires_at": link.expires_at.isoformat(),
        })
    return jsonify({"invitations": result}), 200


def load_invitation_for_invitee(invitation_id):
    """Returns (link, None) or (None, error_response). Only the invited user may answer."""
    link = db.session.get(TrustedLink, invitation_id)
    if link is None:
        return None, (jsonify({"error": "Invitation not found"}), 404)
    if link.contact_user_id != g.user.id:
        return None, access_denied()
    if link.status != "pending":
        return None, (jsonify({"error": "Invitation is not pending"}), 409)
    return link, None


@app.post("/api/invitations/<int:invitation_id>/accept")
@login_required
def accept_invitation(invitation_id):
    link, error = load_invitation_for_invitee(invitation_id)
    if error:
        return error
    link.status = "accepted"
    link.accepted_at = utcnow()
    link.expires_at = None
    # PR4: categories stay unselected; the wearer must choose them
    db.session.commit()
    log_security_event("INVITATION_ACCEPTED", g.user.id)
    return jsonify({"message": "Invitation accepted"}), 200


@app.post("/api/invitations/<int:invitation_id>/decline")
@login_required
def decline_invitation(invitation_id):
    link, error = load_invitation_for_invitee(invitation_id)
    if error:
        return error
    db.session.delete(link)          # PR6: declined invitation is deleted
    db.session.commit()
    log_security_event("INVITATION_DECLINED", g.user.id)
    return jsonify({"message": "Invitation declined"}), 200

In [ ]:
# ---------------- Wearer: manage trusted contacts and sharing ----------------
def load_link_for_wearer(link_id):
    """Returns (link, None) or (None, error_response). Only the wearer who owns the link may manage it (T13)."""
    link = db.session.get(TrustedLink, link_id)
    if link is None:
        return None, (jsonify({"error": "Trusted contact not found"}), 404)
    if link.wearer_id != g.user.id:
        return None, access_denied()
    return link, None


@app.get("/api/trusted-contacts")
@login_required
def list_trusted_contacts():
    links = TrustedLink.query.filter_by(wearer_id=g.user.id).all()
    result = []
    for link in links:
        contact = db.session.get(User, link.contact_user_id)
        result.append({
            "link_id": link.id,
            "contact_name": safe(contact.display_name),
            "status": link.status,
            "categories": categories_of(link),
        })
    return jsonify({"trusted_contacts": result}), 200


@app.put("/api/trusted-contacts/<int:link_id>/categories")
@login_required
def update_sharing_categories(link_id):
    # FR22: the wearer selects, at any time, what each contact can view
    link, error = load_link_for_wearer(link_id)
    if error:
        return error

    if link.status != "accepted":    # PR3: no sharing before acceptance
        return jsonify({"error": "Invitation not accepted yet"}), 409

    data = request.get_json(silent=True)
    if (
        not isinstance(data, dict) or not data
        or any(key not in CATEGORIES for key in data)
        or any(type(value) is not bool for value in data.values())
    ):
        return jsonify({"error": "Invalid categories"}), 400

    for category, value in data.items():
        setattr(link, "share_" + category, value)
    db.session.commit()
    log_security_event("SHARING_SETTINGS_CHANGED", g.user.id)
    return jsonify({"categories": categories_of(link)}), 200


@app.delete("/api/trusted-contacts/<int:link_id>")
@login_required
def remove_trusted_contact(link_id):
    # FR24 (also cancels a pending invitation)
    link, error = load_link_for_wearer(link_id)
    if error:
        return error
    db.session.delete(link)
    db.session.commit()
    log_security_event("TRUSTED_CONTACT_REMOVED", g.user.id)
    return jsonify({"message": "Trusted contact removed"}), 200


# ---------------- Trusted contact: view shared data (FR23, PR10) ----------------
@app.get("/api/shared")
@login_required
def list_shared_wearers():
    links = TrustedLink.query.filter_by(contact_user_id=g.user.id, status="accepted").all()
    result = []
    for link in links:
        wearer = db.session.get(User, link.wearer_id)
        result.append({
            "link_id": link.id,
            "wearer_name": safe(wearer.display_name),
            "available_categories": [c for c, on in categories_of(link).items() if on],
        })
    return jsonify({"wearers": result}), 200


@app.get("/api/shared/<int:link_id>/<category>")
@login_required
def view_shared_data(link_id, category):
    if category not in CATEGORIES:
        return jsonify({"error": "Unknown data category"}), 400

    link = db.session.get(TrustedLink, link_id)
    # missing link, someone else's link, or a removed contact all give the same 403 (PR7)
    if link is None or link.contact_user_id != g.user.id:
        return access_denied()

    if not authorize_access(g.user.id, link.wearer_id, category):    # SR10
        return access_denied()

    log_security_event("SHARED_DATA_ACCESS", g.user.id)               # SR21 (no health values, SR22)
    wearer = db.session.get(User, link.wearer_id)
    return jsonify({
        "wearer_name": safe(wearer.display_name),
        "category": category,
        "data": get_shared_summary(link.wearer_id, category),
    }), 200


# ---------------- Helper for the account-deletion teammate (PR13) ----------------
def remove_user_sharing_data(user_id):
    """Call this inside the account deletion flow: removes the contact ID and every link
    where the account is the wearer or the contact."""
    TrustedLink.query.filter(
        or_(TrustedLink.wearer_id == user_id, TrustedLink.contact_user_id == user_id)
    ).delete(synchronize_session=False)
    ContactProfile.query.filter_by(user_id=user_id).delete(synchronize_session=False)
    db.session.commit()

print("Roles endpoints registered.")

## Tests for the Roles part (positive + negative)

In [ ]:
import uuid, json

def call(client, method, url, body=None):
    token = client.get("/api/auth/csrf-token").get_json()["csrf_token"]
    return client.open(url, method=method, json=body, headers={"X-CSRFToken": token})

def new_user(name):
    client = app.test_client()
    email = f"{name.lower()}-{uuid.uuid4().hex[:8]}@example.com"
    r = call(client, "POST", "/api/auth/register", {
        "display_name": name, "email": email, "password": "TestPass123!",
        "birth_year": 2000, "weight": 60, "privacy_consent": True})
    assert r.status_code == 201, r.get_json()
    r = call(client, "POST", "/api/auth/login", {"email": email, "password": "TestPass123!"})
    assert r.status_code == 200, r.get_json()
    return client

A, B, C = new_user("Alice"), new_user("Bob"), new_user("Carol")   # A = wearer, B = contact, C = outsider

# R1: no session -> 401 (SR11)
anon = app.test_client()
assert anon.get("/api/roles/contact-id").status_code == 401
print("R1 passed: 401 without session")

# R2: contact ID format + never visible to other users (FR17/FR18/SR20/PR11)
a_cid = call(A, "GET", "/api/roles/contact-id").get_json()["contact_id"]
assert len(a_cid) == 8 and all(ch in CONTACT_ID_ALPHABET for ch in a_cid)
b_cid = call(B, "GET", "/api/roles/contact-id").get_json()["contact_id"]
assert a_cid != b_cid
print("R2 passed: 8-char contact ID, one per account")

# R3: invalid / unknown / self invitations (FR19)
assert call(A, "POST", "/api/invitations", {"contact_id": "bad"}).status_code == 400
assert call(A, "POST", "/api/invitations", {"contact_id": "AAAAAAAA"}).status_code == 404
assert call(A, "POST", "/api/invitations", {"contact_id": a_cid}).status_code == 400
print("R3 passed: invalid, unknown and self invitations rejected")

# R4: A invites B; B sees it with A's display name and WITHOUT A's contact ID (FR20/PR11)
r = call(A, "POST", "/api/invitations", {"contact_id": b_cid})
assert r.status_code == 201
link_id = r.get_json()["link_id"]
pending = call(B, "GET", "/api/invitations/pending")
assert pending.status_code == 200 and pending.get_json()["invitations"][0]["wearer_name"] == "Alice"
assert a_cid not in pending.get_data(as_text=True)
assert call(A, "POST", "/api/invitations", {"contact_id": b_cid}).status_code == 409
print("R4 passed: pending invitation visible, contact ID hidden, duplicate blocked")

# R5: another user cannot answer B's invitation (IDOR) -> 403 (SR12)
assert call(C, "POST", f"/api/invitations/{link_id}/accept").status_code == 403
print("R5 passed: 403 for a user who is not the invitee")

# R6: no access before acceptance (PR3)
assert call(B, "GET", f"/api/shared/{link_id}/heart_rate_summary").status_code == 403
assert call(A, "PUT", f"/api/trusted-contacts/{link_id}/categories", {"alerts": True}).status_code == 409
print("R6 passed: no access / no category selection before acceptance")

# R7: after acceptance nothing is selected by default (PR4)
assert call(B, "POST", f"/api/invitations/{link_id}/accept").status_code == 200
assert all(v is False for v in call(A, "GET", "/api/trusted-contacts").get_json()["trusted_contacts"][0]["categories"].values())
assert call(B, "GET", f"/api/shared/{link_id}/heart_rate_summary").status_code == 403
print("R7 passed: accepted contact starts with no categories")

# R8: only the selected category is visible (FR22/FR23/PR10)
assert call(A, "PUT", f"/api/trusted-contacts/{link_id}/categories", {"heart_rate_summary": True}).status_code == 200
assert call(B, "GET", f"/api/shared/{link_id}/heart_rate_summary").status_code == 200
assert call(B, "GET", f"/api/shared/{link_id}/sleep_summary").status_code == 403
assert call(B, "GET", f"/api/shared/{link_id}/raw_readings").status_code == 400
print("R8 passed: only selected categories are readable")

# R9: contact cannot change or remove sharing (T13 elevation of privilege)
assert call(B, "PUT", f"/api/trusted-contacts/{link_id}/categories", {"sleep_summary": True}).status_code == 403
assert call(B, "DELETE", f"/api/trusted-contacts/{link_id}").status_code == 403
assert call(C, "GET", f"/api/shared/{link_id}/heart_rate_summary").status_code == 403
print("R9 passed: contact/outsider cannot modify settings or read others' links")

# R10: invalid category payloads rejected (input validation)
for bad in ({}, {"alerts": "yes"}, {"email": True}, {"alerts": 1}):
    assert call(A, "PUT", f"/api/trusted-contacts/{link_id}/categories", bad).status_code == 400
print("R10 passed: invalid category payloads rejected")

# R11: deselect / remove -> denied from the NEXT request (PR7)
assert call(A, "PUT", f"/api/trusted-contacts/{link_id}/categories", {"heart_rate_summary": False}).status_code == 200
assert call(B, "GET", f"/api/shared/{link_id}/heart_rate_summary").status_code == 403
call(A, "PUT", f"/api/trusted-contacts/{link_id}/categories", {"heart_rate_summary": True})
assert call(B, "GET", f"/api/shared/{link_id}/heart_rate_summary").status_code == 200
assert call(A, "DELETE", f"/api/trusted-contacts/{link_id}").status_code == 200
assert call(B, "GET", f"/api/shared/{link_id}/heart_rate_summary").status_code == 403
print("R11 passed: deselect/remove takes effect on the next request")

# R12: decline deletes the invitation (PR6)
c_cid = call(C, "GET", "/api/roles/contact-id").get_json()["contact_id"]
inv = call(A, "POST", "/api/invitations", {"contact_id": c_cid}).get_json()["link_id"]
assert call(C, "POST", f"/api/invitations/{inv}/decline").status_code == 200
with app.app_context():
    assert db.session.get(TrustedLink, inv) is None
print("R12 passed: declined invitation deleted")

# R13: expired invitation (7 days) is deleted (PR5)
inv = call(A, "POST", "/api/invitations", {"contact_id": c_cid}).get_json()["link_id"]
with app.app_context():
    db.session.get(TrustedLink, inv).expires_at = utcnow() - timedelta(minutes=1)
    db.session.commit()
assert call(C, "GET", "/api/invitations/pending").get_json()["invitations"] == []
with app.app_context():
    assert db.session.get(TrustedLink, inv) is None
print("R13 passed: expired invitation deleted")

# R14: audit events were written (SR21)
with app.app_context():
    events = {row.event_type for row in SecurityLog.query.all()}
for expected in ("INVITATION_SENT", "INVITATION_ACCEPTED", "INVITATION_DECLINED",
                 "SHARING_SETTINGS_CHANGED", "TRUSTED_CONTACT_REMOVED", "SHARED_DATA_ACCESS", "AUTHORIZATION_FAILED"):
    assert expected in events, expected
print("R14 passed: audit events logged")
print("\nAll Roles tests passed.")